DL2A_fashion_mnist_2025.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/10zM8vz1VezS4-llyLhmvJlf4uOQIu7z5

# Fashion-MNIST dataset & Feed-Forward Neural Networks with PyTorch

***

# Foreword
This notebook serves as **a template** for your experiments and explorations. You are encouraged not only to write the necessary code to complete the assigned tasks but also **to document your work clearly**. This includes **adding Markdown cells with your explanations, conclusions, and figures**. *The goal is for you to make this notebook into a nice report!*

Here are **a few key points** to keep in mind:

- Make sure you understand the code you write.

- Clearly explain your results and observations.

- Write clean, well-documented code.

- Avoid copy-pasting the same code multiple times — instead, define reusable functions!

# Main Objective: Image Classification

This lab introduces **feed-forward neural networks using PyTorch**, with a focus on **image classification**. You’ll work with the Fashion-MNIST dataset (see [Fashion-MNIST GitHub](https://github.com/zalandoresearch/fashion-mnist) for more details), which consists of:
- 60,000 training images
- 10,000 testing images
- Each image is 28×28 pixels, totaling 784 features per image
- The images are flattened into vectors of length 784 before being passed to the network

During this session, you’ll build and train several feed-forward models, starting with simple architectures and gradually increasing their complexity.

*!! First load and test Python and PyTorch. Your notebook is supposed to work with Python 3 (see the top right corner of the notebook).*

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
import torch as th
import torch.nn as nn
import numpy as np
import pickle
import math, gzip, time
import matplotlib
import matplotlib.pyplot as plt
# %matplotlib inline
# %config InlineBackend.figure_formats=['svg']
# %config InlineBackend.figure_format = 'svg'
print(th.__version__) # should be greater or equal to 1.0

# Dataset

As already mentioned, we will use the Fashion-MNIST dataset for image classification. You can download it via torchvision, but, for simplicity, a preprocessed pickle file can be dowloaded using the link below.

## Download the dataset

**If you are using Colab --- prefered option**

In [ ]:
import gdown
url = 'https://drive.google.com/uc?id=1qx-a3KjzX2W66Aq84tnjGRUa10CTxOJN'
output = 'fashion-mnist.pk.gz'
gdown.download(url, output, quiet=False)

If you are using Jupyter

In [ ]:
# To download the file with wget:
# ! wget -nc --no-check-certificate "https://drive.usercontent.google.com/download?id=1qx-a3KjzX2W66Aq84tnjGRUa10CTxOJN&confirm=t" -O "./fashion-mnist.pk.gz"

Check that the file has been downloaded (~43MB):

In [ ]:
import os
print(os.path.getsize("fashion-mnist.pk.gz") / 1e6, "Mo")   # doit faire ~43 Mo

### Load the Dataset

In [ ]:
fp = gzip.open('fashion-mnist.pk.gz','rb')
allXtrain, allYtrain, Xtest, Ytest, classlist  = pickle.load(fp)
allXtrain=allXtrain/255

**Important**: As mentioned before, the dataset is split in two parts, the training set and the test set. For thorough study and evaluation of machine learning models, **a good practice is to actually split the data into 3 sets:**

- **Training set**: to learn model parameters
- **Validation set**: to tune the hyperparameters and some design choices (the number and the size of the hidden layers, the dropout probability, ...)
- **Test set**: to evaluate the final model.

### Training / validation sets

To **speed up** training during this lab, we'll use only the **first 20 000 samples for training** and take out another **10 000 for validation**.

In [ ]:
Xtrain, Ytrain  = allXtrain[:20000], allYtrain[:20000]
Xvalid, Yvalid  = allXtrain[20000:30000], allYtrain[20000:30000]
print("Training   shape:" ,Xtrain.shape)
print("Validation shape:" ,Xvalid.shape)

To better understand the data you are dealing with it is crucial to explore it! In this case this is the training set stored in the variables **Xtrain and Ytrain**.

- Look at the dimension and type of the tensors stored in these variables.
- Print the classlist variable.
- Look at some examples to check consistency.

For that purpose you can **visualize** training examples like this:

In [ ]:
indices = np.arange(len(Xtrain))
nimages = 6
sel = np.random.choice(indices, nimages)
fig, axs = plt.subplots(1,nimages,figsize=(10,2))
for i,idx in enumerate(sel):
    axs[i].imshow(Xtrain[idx].numpy().reshape(28,28) , matplotlib.pyplot.cm.gray)

# print class list variable
print(classlist)

# Feedforward Neural Network

## PyTorch Sequential Module

In PyTorch, a feedforward neural network can be built using the **nn.Sequential container**. This allows you to stack layers in order, where:

- The first layer takes the input tensor.

- Each subsequent layer receives the output from the previous one.

This makes model definition concise and easy to read.


### IMPORTANT: Tensor Shapes in PyTorch -- Online Mode vs. Mini-Batch

PyTorch expects input data in the form of a 2D tensor with shape **(B, D)**, where:

- B is the batch size (number of examples),

- D is the input dimension (e.g., number of pixels in an image).

There are two common ways to feed data:

- **Online mode:** B = 1, for training or predicting one example at a time.

- **Mini-batch mode:** B > 1, for processing multiple examples simultaneously.


Note: if "B= Number of training example", you take all the training data at once. It is sometime called full-batch training.  


Even if you're passing a single example (B = 1), the input must still have two dimensions.
For example:

- A batch of 200 images: shape = (200, 784)
- A single image: shape = (1, 784) (not just (784,)!)
    
        

## Shallow Neural Network - no hidden layers

Let’s start with a very simple model:

- No hidden layers

- Just one input layer and one output layer, i.e., one linear transformation from 784 input pixels to 10 output classes

**Your task is to:**

- Propose an implementation of this linear model. Use **nn.Sequential** module as container to define your model.

- Train the model to classify images, i.e., maximize the log-likelihood of the correct class, or equivalently,   minimize the **Negative Log-Likelihood** (NLL) loss function (discussed in TD).

In PyTorch [documentation of the NNet package](https://pytorch.org/docs/stable/nn.html) you will find a section on the loss functions. Browse this section to find the appropriate one. It will also tell you how to design the output of the model.

In [ ]:
D_in = 784
D_out= 10

## -----------> TODO : replace "None" <----------------
model = nn.Sequential(nn.Linear(D_in,D_out),nn.LogSoftmax(dim=1)) #Ici on definit notre model comme étant la composition de deux fonction une linearisation et une logproba
loss_function = nn.NLLLoss() #defini notre loss function

preds = model(Xtrain)
loss=loss_function(preds,Ytrain)

In [ ]:
print("Shape:",preds.shape)
print("1st prediction:",preds[0])
print("Good answer:",Ytrain[0])
print(classlist)

- Test the code on **a minibatch** of B examples

*NB. Don't confuse **minibatch** with **batchnorm** --- **Minibatch** is a training strategy where you split your full dataset into smaller minibatches (e.g., 32 or 64 samples per batch), and update model weights after computing gradients on each minibatch. And **BatchNorm** is a normalization layer added to neural networks to stabilize and speed up training by standardizing layer inputs.*



The code below corresponds to a prediction on a single image and then on 3 images.
*NB. Even when predicting a single image, the input should be 2D: use Xtrain[i:i+1] instead of Xtrain[i].*

- Look at the results, their shapes and values. Is it consistent with what you expect ?
- What do you think about the output ?

In [ ]:
B=1
i = 0
pred = model(Xtrain[i:i+B])
loss=loss_function(pred,Ytrain[i:i+B])
print(loss)
print(pred)

In [ ]:
B=3
i = 0
pred = model(Xtrain[i:i+B])
loss=loss_function(pred,Ytrain[i:i+B])
print(loss)
print(pred)

- Do the same with the loss function.

In [ ]:
## -----------> TODO : add loss function <----------------
# La NLLLoss attend :
#   - en entree  : des LOG-probabilites, de shape (B, C)   <- d'ou le LogSoftmax final
#   - en cible   : des INDICES de classe (pas du one-hot), de shape (B,), type long
# Elle renvoie un SCALAIRE : la moyenne sur le batch de -log p(classe correcte).
for B in [1, 3, 200]:
    pred = model(Xtrain[0:B])
    loss = loss_function(pred, Ytrain[0:B])
    print(f"B = {B:3d} | pred {tuple(pred.shape)} | loss = {loss.item():.4f} (scalaire)")

# Verification a la main sur un seul exemple : la loss vaut bien -log p(bonne classe)
pred = model(Xtrain[0:1])
print("\n-log p(classe correcte) calcule a la main :", -pred[0, Ytrain[0]].item())
print("valeur renvoyee par NLLLoss              :", loss_function(pred, Ytrain[0:1]).item())

# Avant entrainement les 10 classes sont ~equiprobables, donc la loss doit
# valoir environ -log(1/10) = log(10) = 2.30. C'est LE chiffre de reference
# pour verifier qu'un modele demarre correctement.
print("\nloss attendue au hasard : log(10) =", math.log(10))
print("loss mesuree sur 1000 exemples :", loss_function(model(Xtrain[:1000]), Ytrain[:1000]).item())

### Training function

We now write **a generic training loop**. The code should generic so that it can handle different models and parameters.

**Your tasks:**

- Write a code to train a model with.
  - Optimizer: SGD -- stochastic gradient descent (you've seen it in TD)
  - Learning rate: 0.001
  - Epochs: 200

- Try different values of the learning rate.

Once you make it work, wrap your code in a function that you can reuse.

- Write a function that wraps everything you need to train the model and display results.
- Test is it with a different model.

Your function should:

- Track training and validation loss
- Report classification accuracy
- Show training curves


**NB on the structure of the output:** When you pass a batch of images through your model like this *output = model(X)*, you get a 2D tensor as output. In this tensor:

- each row corresponds to one image from your batch.
- each column corresponds to one of the possible labels (here 10, for the 10 Fashion-MNIST classes).

For example, if X contains a batch of 64 images, the output tensor will have shape: *torch.Size([64, 10])* Each value *output[i, j]* is the log-probability that the i-th image belongs to class j.

In [ ]:
## -----------> TODO generic training loop <----------------
# La boucle minimale, en full-batch. Les 4 lignes du coeur sont TOUJOURS
# les memes, quel que soit le modele :
NEpochs = 20
model = nn.Sequential(nn.Linear(D_in, D_out), nn.LogSoftmax(dim=1))
optimizer = th.optim.SGD(model.parameters(), lr=0.1)

for i in range(NEpochs):
    Y_pred = model(Xtrain)                    # 1. forward
    loss = loss_function(Y_pred, Ytrain)      # 2. loss
    loss.backward()                           # 3. backward  (calcule .grad)
    optimizer.step()                          # 4. update    (utilise .grad)
    optimizer.zero_grad()                     # 5. reset     (sinon accumulation)
print("loss finale :", loss.item())

## -----------> TODO: train function <----------------
# On generalise maintenant cette boucle dans une fonction reutilisable.
# Par rapport a la version minimale on ajoute :
#   - le suivi de la loss de VALIDATION (pas seulement celle de train :
#     c'est le seul moyen de voir l'overfitting) ;
#   - l'accuracy sur train et validation ;
#   - model.train() / model.eval(), indispensables des qu'il y a du Dropout
#     ou de la BatchNorm (ces couches n'ont pas le meme comportement
#     en entrainement et en inference) ;
#   - th.no_grad() en evaluation : on ne construit pas le graphe, c'est
#     plus rapide et ca economise beaucoup de memoire ;
#   - le renvoi d'un dictionnaire d'historique, pour pouvoir comparer
#     plusieurs runs sur une meme figure.

def evaluate(model, X, Y, loss_function):
    """Renvoie (loss, accuracy) d'un modele sur un jeu de donnees."""
    model.eval()                              # desactive Dropout, fige BatchNorm
    with th.no_grad():                        # pas de graphe de calcul en inference
        out = model(X)
        loss = loss_function(out, Y).item()
        pred = out.argmax(dim=1)              # la classe de log-proba maximale
        acc = (pred == Y).float().mean().item()
    return loss, acc

In [ ]:
def train_model(model, loss_function, optimizer, Xtrain, Ytrain, NEpochs,
                Xvalid=None, Yvalid=None, batch_size=None, plot=True, label=None,
                verbose=True):
    """Entraine un modele et renvoie l'historique.

    Args:
        batch_size : None -> full-batch (une mise a jour par epoch)
                     n    -> mini-batch de taille n (donnees remelangees a chaque epoch)
        Xvalid/Yvalid : si None, on retombe sur les variables globales du notebook.
    Returns:
        dict avec les cles 'train_loss', 'valid_loss', 'train_acc', 'valid_acc', 'time'
    """
    if Xvalid is None:
        Xvalid, Yvalid = globals()['Xvalid'], globals()['Yvalid']

    hist = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
    N = len(Xtrain)
    t0 = time.time()

    for epoch in range(NEpochs):
        model.train()                         # active Dropout, met BatchNorm en mode batch
        if batch_size is None:                # ---- full-batch ----
            Y_pred = model(Xtrain)
            loss = loss_function(Y_pred, Ytrain)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        else:                                 # ---- mini-batch ----
            perm = th.randperm(N)             # nouveau decoupage a chaque epoch
            for k in range(0, N, batch_size):
                idx = perm[k:k + batch_size]
                loss = loss_function(model(Xtrain[idx]), Ytrain[idx])
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        tr_loss, tr_acc = evaluate(model, Xtrain, Ytrain, loss_function)
        va_loss, va_acc = evaluate(model, Xvalid, Yvalid, loss_function)
        hist['train_loss'].append(tr_loss); hist['train_acc'].append(tr_acc)
        hist['valid_loss'].append(va_loss); hist['valid_acc'].append(va_acc)

        if verbose and (epoch % max(1, NEpochs // 10) == 0 or epoch == NEpochs - 1):
            print(f"epoch {epoch:4d} | train loss {tr_loss:.4f} acc {tr_acc:.4f}"
                  f" | valid loss {va_loss:.4f} acc {va_acc:.4f}")

    hist['time'] = time.time() - t0
    if plot:
        plot_history(hist, title=label or "")
    return hist

In [ ]:
def plot_history(hist, title=""):
    """Trace loss et accuracy, train et validation sur le meme graphe."""
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    axs[0].plot(hist['train_loss'], color='red', label='train')
    axs[0].plot(hist['valid_loss'], color='orange', linestyle='--', label='validation')
    axs[0].set_title('Loss'); axs[0].set_xlabel('epoch'); axs[0].set_ylabel('NLL')
    axs[0].grid(True, linestyle='--', alpha=.6); axs[0].legend()

    axs[1].plot(hist['train_acc'], color='green', label='train')
    axs[1].plot(hist['valid_acc'], color='darkgreen', linestyle='--', label='validation')
    axs[1].set_title('Accuracy'); axs[1].set_xlabel('epoch'); axs[1].set_ylabel('accuracy')
    axs[1].grid(True, linestyle='--', alpha=.6); axs[1].legend()

    fig.suptitle(f"{title}   (temps : {hist['time']:.1f} s)")
    plt.tight_layout(); plt.show()

In [ ]:
model = nn.Sequential(nn.Linear(D_in, D_out), nn.LogSoftmax(dim=1))
optimizer = th.optim.SGD(model.parameters(), lr=0.1)
hist_lin = train_model(model, loss_function, optimizer, Xtrain, Ytrain, 200,
                       label="Modele lineaire (pas de couche cachee), lr=0.1")

Le modele lineaire plafonne autour de **83 % d'accuracy** en validation.
C'est deja beaucoup mieux que le hasard (10 %), ce qui montre que Fashion-MNIST
est en grande partie separable lineairement - mais on va faire mieux.

Un point de vocabulaire important : ce modele "sans couche cachee" est
exactement une **regression logistique multiclasse** (softmax regression).
`Linear + LogSoftmax + NLLLoss` est la version multiclasse du
`Linear + Sigmoid + BCELoss` du TP precedent.

In [ ]:
print("taille du jeu de validation :", len(Xvalid))

## Neural Network with one hidden layer

Now we have a function to train a neural model and evaluate the training process. This is your starting point for exploration of **different network architectures**.

Your next architecture will be a network with:

    One hidden layer of size 50

    Sigmoid activation

Write the model using the **nn.Sequential** module, and train it:

- for 30 epochs with learning rates 0.001 and 0.0001
- for 50 epochs and the same learning rates
- What do you observe ? How do learning rates and number of epochs affect your results ?

In [ ]:
## -----------> TODO <----------------
# On definit d'abord une petite fabrique de modeles, pour eviter de recopier
# la meme nn.Sequential a chaque experience (et pour garantir que seule la
# chose qu'on etudie change).

def make_mlp(hidden_sizes=(50,), activation=nn.ReLU, dropout=0.0, batchnorm=False):
    """Construit un MLP  784 -> hidden_sizes -> 10  suivi d'un LogSoftmax.

    L'ordre a l'interieur d'un bloc cache est : Linear -> BatchNorm -> activation -> Dropout.
    (BatchNorm AVANT l'activation : c'est l'ordre de l'article original.)
    """
    layers, d_prev = [], D_in
    for h in hidden_sizes:
        layers.append(nn.Linear(d_prev, h))
        if batchnorm:
            layers.append(nn.BatchNorm1d(h))
        layers.append(activation())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        d_prev = h
    layers += [nn.Linear(d_prev, D_out), nn.LogSoftmax(dim=1)]
    return nn.Sequential(*layers)

In [ ]:
def run(model, lr=0.1, momentum=0.0, NEpochs=50, batch_size=None, label="", plot=True):
    """Raccourci : cree l'optimiseur SGD et lance l'entrainement."""
    optimizer = th.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    return train_model(model, loss_function, optimizer, Xtrain, Ytrain, NEpochs,
                       batch_size=batch_size, label=label, plot=plot, verbose=False)

In [ ]:
# --- L'experience demandee : une couche cachee de 50, activation SIGMOID ---
res_sigmoid = {}
for NEpochs in (30, 50):
    for lr in (0.001, 0.0001):
        th.manual_seed(0)
        m = make_mlp((50,), activation=nn.Sigmoid)
        h = run(m, lr=lr, NEpochs=NEpochs, plot=False)
        res_sigmoid[(NEpochs, lr)] = h
        print(f"Sigmoid | {NEpochs:2d} epochs | lr={lr:<7} -> "
              f"valid acc = {h['valid_acc'][-1]:.4f} | valid loss = {h['valid_loss'][-1]:.4f}")

plt.figure(figsize=(7, 4))
for (NEpochs, lr), h in res_sigmoid.items():
    plt.plot(h['valid_acc'], label=f"{NEpochs} ep, lr={lr}")
plt.xlabel("epoch"); plt.ylabel("accuracy validation")
plt.title("Une couche cachee (50, Sigmoid) : effet de lr et du nb d'epochs")
plt.legend(); plt.grid(alpha=.3); plt.show()

**Observations.**

Avec `lr = 0.0001` le modele ne bouge quasiment pas : en 50 epochs de *full-batch*
il n'y a que 50 mises a jour au total, chacune minuscule. L'accuracy reste proche
du niveau du hasard. Avec `lr = 0.001` c'est un peu mieux mais toujours tres lent.

C'est le point cle de cette section : en **full-batch**, une epoch = **une seule**
mise a jour des parametres. Un learning rate qui semble "raisonnable" en
mini-batch (1e-3, valeur typique avec Adam) est ici beaucoup trop petit. Il faut
soit monter le learning rate (0.1 - 0.5), soit passer en mini-batch (section
plus bas), soit les deux.

Second effet visible : le **Sigmoid sature**. Sa derivee vaut au maximum 0.25 et
tend vers 0 des que l'entree s'eloigne de 0, donc le gradient qui remonte vers
la premiere couche est fortement attenue. C'est le probleme du *vanishing
gradient*, et c'est exactement ce que la ReLU corrige.

On refait donc les memes runs avec un learning rate adapte :

In [ ]:
for lr in (0.01, 0.1, 0.5):
    th.manual_seed(0)
    m = make_mlp((50,), activation=nn.Sigmoid)
    h = run(m, lr=lr, NEpochs=50, plot=False)
    print(f"Sigmoid | 50 epochs | lr={lr:<5} -> valid acc = {h['valid_acc'][-1]:.4f}")

## SGD with momentum

Up to now, you have trained your NNets using the SGD optimizer. This works fine but in general SGD can be slow to converge, especially when the loss surface has gorges with steep sides, i.e., regions where the gradient changes steeply in some directions and slowly in others. This can cause the optimization to zigzag and take longer to reach a minimum.

A way to overcome this problem is to use **momentum** -- a parameter in SGD optimizer that helps accelerate training, especially in tricky regions. Basically, instead of updating parameters purely based on the current gradient, we also consider the previous updates. This creates a sort of "velocity" that helps smooth out the path and push through small oscillations.

**Your task:**

- Add the moment term to the optimizer (try 0.9)
- See the impact on the training, with different values.

In [ ]:
## -----------> TODO <----------------
# On compare plusieurs valeurs de momentum, tout le reste etant identique
# (meme seed, donc meme initialisation des poids).
hist_momentum = {}
for mom in (0.0, 0.2, 0.5, 0.7, 0.9, 0.99):
    th.manual_seed(0)
    m = make_mlp((50,), activation=nn.ReLU)
    hist_momentum[mom] = run(m, lr=0.1, momentum=mom, NEpochs=50, plot=False)
    print(f"momentum = {mom:<5} -> valid acc = {hist_momentum[mom]['valid_acc'][-1]:.4f}")

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for mom, h in hist_momentum.items():
    axs[0].plot(h['train_loss'], label=f"m = {mom}")
    axs[1].plot(h['valid_acc'], label=f"m = {mom}")
axs[0].set_title("Loss d'entrainement"); axs[0].set_xlabel("epoch"); axs[0].set_yscale('log')
axs[1].set_title("Accuracy validation"); axs[1].set_xlabel("epoch")
for a in axs: a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Ce que fait le momentum.** Au lieu de suivre le gradient courant, on suit une
moyenne exponentielle des gradients passes :

$$v_{t} = \\beta\\, v_{t-1} + g_t \\qquad \\theta_{t+1} = \\theta_t - \\eta\\, v_t$$

Les composantes du gradient qui pointent toujours dans la meme direction
s'additionnent (on accelere), celles qui changent de signe a chaque pas
s'annulent (on amortit les oscillations dans les "gorges" de la surface de
loss). Le facteur d'acceleration effectif est de l'ordre de $1/(1-\\beta)$ :
avec $\\beta = 0.9$ on avance environ 10 fois plus vite a learning rate egal.

**Observation.** 0.9 est nettement meilleur que 0 : c'est la valeur par defaut
utilisee partout, et pour une bonne raison. En revanche 0.99 devient instable
ici, parce que `lr = 0.1` combine a une acceleration x100 fait des pas beaucoup
trop grands. Regle pratique : **quand on augmente le momentum, il faut baisser
le learning rate.**

## From Sigmoid to ReLU activation

**Your task:**

- Consider lr=0.0001 and train your model with a ReLU activation.
- Compare the results to the Sigmoid.

In [ ]:
## -----------> TODO <----------------
# Comparaison a strictement identique : meme architecture, meme seed, meme lr,
# seule l'activation change.
for lr in (0.0001, 0.1):
    plt.figure(figsize=(7, 4))
    for name, act in [("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU)]:
        th.manual_seed(0)
        m = make_mlp((50,), activation=act)
        h = run(m, lr=lr, NEpochs=100, plot=False)
        plt.plot(h['valid_acc'], label=f"{name} (fin : {h['valid_acc'][-1]:.3f})")
        print(f"lr={lr:<7} | {name:8s} -> valid acc = {h['valid_acc'][-1]:.4f}")
    plt.xlabel("epoch"); plt.ylabel("accuracy validation")
    plt.title(f"Sigmoid vs Tanh vs ReLU, lr = {lr}")
    plt.legend(); plt.grid(alpha=.3); plt.show()
    print()

**Conclusion.** A `lr = 0.0001` *aucune* des trois activations n'apprend quoi que
ce soit : le probleme vient du learning rate, pas de l'activation. C'est un piege
classique - avant de conclure "l'architecture A est meilleure que B", il faut
s'assurer que les deux sont entrainees dans un regime ou elles apprennent
reellement.

A learning rate correct, la hierarchie attendue apparait :

| Activation | Derivee max | Probleme |
|---|---|---|
| Sigmoid $\\sigma(x)$ | 0.25 | sature des deux cotes, sortie non centree en 0 -> gradient qui s'evanouit |
| Tanh | 1.0 | sature aussi, mais au moins centree en 0 |
| ReLU $\\max(0,x)$ | 1 (si $x>0$) | ne sature pas a droite ; risque de "neurones morts" si $x<0$ en permanence |

La ReLU gagne surtout parce que sa derivee vaut exactement 1 sur toute la partie
active : le gradient traverse les couches **sans etre attenue**, ce qui est ce
qui rend les reseaux profonds entrainables. C'est aussi la moins chere a calculer.

## Impact of the hidden layer size

Run experiments with different hidden layer sizes.

    Try: 50,100,150,200 and 250.

- What do you observe ?

In [ ]:
## -----------> TODO <----------------
# ATTENTION a la rigueur experimentale : pour comparer des tailles de couche
# cachee, il faut garder TOUT le reste constant, y compris le nombre d'epochs
# (50 partout ici) et la seed.
hist_H = {}
for H in (50, 100, 150, 200, 250):
    th.manual_seed(0)
    m = make_mlp((H,), activation=nn.ReLU)
    hist_H[H] = run(m, lr=0.1, momentum=0.9, NEpochs=50, plot=False)
    nparam = sum(p.numel() for p in m.parameters())
    print(f"H = {H:3d} | {nparam:7d} parametres | train acc = {hist_H[H]['train_acc'][-1]:.4f}"
          f" | valid acc = {hist_H[H]['valid_acc'][-1]:.4f}"
          f" | ecart = {hist_H[H]['train_acc'][-1] - hist_H[H]['valid_acc'][-1]:+.4f}")

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for H, h in hist_H.items():
    axs[0].plot(h['train_acc'], label=f"H = {H}")
    axs[1].plot(h['valid_acc'], label=f"H = {H}")
axs[0].set_title("Accuracy TRAIN"); axs[1].set_title("Accuracy VALIDATION")
for a in axs: a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 4))
Hs = list(hist_H.keys())
plt.plot(Hs, [hist_H[H]['train_acc'][-1] for H in Hs], 'o-', label="train")
plt.plot(Hs, [hist_H[H]['valid_acc'][-1] for H in Hs], 's-', label="validation")
plt.xlabel("taille de la couche cachee H"); plt.ylabel("accuracy finale")
plt.title("Capacite du modele vs generalisation"); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Observations.** L'accuracy de *train* augmente de facon monotone avec `H` :
plus le modele a de parametres, plus il peut coller aux donnees d'entrainement.
L'accuracy de *validation*, elle, progresse fortement de 50 a ~150 puis
**plafonne**. L'ecart train - validation se creuse : c'est la signature du
debut de l'**overfitting**.

C'est la lecon a retenir : augmenter la capacite ne fait pas gagner
indefiniment. Passe un certain point, la capacite supplementaire sert a
memoriser le bruit du jeu d'entrainement plutot qu'a apprendre des motifs
generalisables. C'est precisement ce que le Dropout et le weight decay vont
venir contrer.

## Deeper network -- two hidden layers

Now we add one more hidden layer to our architecture.

To start with, consider:

    Two hidden layers of size 50

    Activation: ReLU
    
    Learning rate: 0.0001

    Train for 100 epochs
    

In [ ]:
## -----------> TODO <----------------

model=nn.Sequential(nn.Linear(D_in,50),nn.ReLU(),nn.Linear(50,50),nn.ReLU(),nn.Linear(50,10),nn.LogSoftmax(dim=1))
optimizer = th.optim.SGD(model.parameters(), lr=0.1,momentum=0.9)
train_model(model,loss_function,optimizer,Xtrain,Ytrain,100)

    Try different dropout values (e.g., 0.3, 0.5)
    
NB: When you apply dropout with a certain probability **p**, each neuron in a given layer has a chance **p** of being dropped during training. This forces the network to not rely too heavily on any single neuron, which improves generalization. The value 0.3 is used for mild regularization, i.e., when you have a small or simple network; 0.5 corresponds to stronger regularization and is a typically used for deeper or overfitting networks.       

In [ ]:
## -----------> TODO <----------------
hist_drop = {}
for p in (0.0, 0.3, 0.5):
    th.manual_seed(0)
    m = make_mlp((50, 50), activation=nn.ReLU, dropout=p)
    hist_drop[p] = run(m, lr=0.1, momentum=0.9, NEpochs=100, plot=False)
    h = hist_drop[p]
    print(f"dropout p = {p} | train acc = {h['train_acc'][-1]:.4f}"
          f" | valid acc = {h['valid_acc'][-1]:.4f}"
          f" | ecart = {h['train_acc'][-1] - h['valid_acc'][-1]:+.4f}")

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for p, h in hist_drop.items():
    axs[0].plot(h['train_loss'], label=f"p = {p} (train)")
    axs[0].plot(h['valid_loss'], '--', label=f"p = {p} (valid)")
    axs[1].plot(h['valid_acc'], label=f"p = {p}")
axs[0].set_title("Loss : l'ecart train/valid = overfitting"); axs[0].set_yscale('log')
axs[1].set_title("Accuracy validation")
for a in axs: a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Lecture des courbes.** Sans dropout, la loss de train continue de descendre
alors que la loss de *validation* remonte : le modele memorise. Avec dropout,
la loss de train est plus haute (c'est normal, le reseau est handicape pendant
l'entrainement) mais l'ecart entre les deux courbes se resserre.

`p = 0.5` sur un reseau aussi petit (50 neurones par couche) est trop agressif :
on ne garde que 25 neurones en moyenne, le modele sous-apprend. `p = 0.3` est le
meilleur compromis ici. Regle generale : plus le reseau est large, plus on peut
mettre un dropout fort.

Play with different hyperparameters:

    Larger hidden layers (double it for example)

    Adding a third hidden layer

In [ ]:
## -----------> TODO <----------------
architectures = {
    "1 x 50":            (50,),
    "2 x 50":            (50, 50),
    "2 x 100 (double)":  (100, 100),
    "3 x 100":           (100, 100, 100),
    "3 x 200":           (200, 200, 200),
}
hist_arch = {}
for name, sizes in architectures.items():
    th.manual_seed(0)
    m = make_mlp(sizes, activation=nn.ReLU)
    hist_arch[name] = run(m, lr=0.1, momentum=0.9, NEpochs=100, plot=False)
    h = hist_arch[name]
    print(f"{name:18s} | {sum(p.numel() for p in m.parameters()):8d} params"
          f" | train {h['train_acc'][-1]:.4f} | valid {h['valid_acc'][-1]:.4f}"
          f" | {h['time']:5.1f} s")

plt.figure(figsize=(7, 4))
for name, h in hist_arch.items():
    plt.plot(h['valid_acc'], label=name)
plt.xlabel("epoch"); plt.ylabel("accuracy validation")
plt.title("Profondeur et largeur du reseau"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.show()

**Observation.** Passer de 1 a 2 couches cachees aide nettement. Au-dela, le
gain devient marginal *et* l'entrainement devient plus difficile : les reseaux
profonds entraines en SGD simple souffrent de gradients mal conditionnes. C'est
exactement le probleme que la BatchNorm resout, d'ou la section suivante.

## Add Batch Normalization

BatchNorm helps deeper networks train faster and more reliably.

**Your task:**

- Build a neural network with 2 hidden layers.
- Insert Batch Normalization block where it is useful.
- Train the model and compare the optimization speed.

In [ ]:
## -----------> TODO <----------------
# BatchNorm normalise les activations d'une couche sur la dimension du BATCH :
#   x_hat = (x - mu_batch) / sqrt(var_batch + eps)   puis   y = gamma * x_hat + beta
# gamma et beta sont APPRIS (le reseau peut donc annuler la normalisation s'il
# le veut). En inference, mu et var sont remplaces par des moyennes glissantes
# accumulees pendant l'entrainement -> d'ou l'importance de model.eval().
#
# Placement : entre Linear et l'activation. Note qu'en 1D on utilise
# nn.BatchNorm1d(H) ; sur des images (TP suivant) ce sera nn.BatchNorm2d(C).
#
# ATTENTION : la BatchNorm ne fonctionne pas en full-batch pur avec un seul
# echantillon, et surtout elle n'a d'interet qu'en mini-batch. On compare donc
# ici en mini-batch (batch_size = 200).
for bn in (False, True):
    th.manual_seed(0)
    m = make_mlp((100, 100), activation=nn.ReLU, batchnorm=bn)
    h = run(m, lr=0.1, momentum=0.9, NEpochs=30, batch_size=200, plot=False)
    print(f"batchnorm = {str(bn):5s} | valid acc = {h['valid_acc'][-1]:.4f}"
          f" | acc apres 5 epochs = {h['valid_acc'][4]:.4f} | {h['time']:.1f} s")
    plt.plot(h['valid_acc'], label=f"BatchNorm = {bn}")
plt.xlabel("epoch"); plt.ylabel("accuracy validation")
plt.title("Effet de la BatchNorm (2 x 100, mini-batch 200)")
plt.legend(); plt.grid(alpha=.3); plt.show()

**Ce qu'on doit voir.** La BatchNorm ne change pas beaucoup l'accuracy finale
sur un probleme aussi simple, mais elle fait converger **beaucoup plus vite** :
comparer l'accuracy au bout de 5 epochs. Elle permet aussi d'utiliser des
learning rates plus eleves sans diverger.

Deux effets secondaires a connaitre :
1. elle a un **effet regularisant** (le bruit d'estimation de mu/var sur chaque
   batch agit comme du bruit d'entrainement), donc elle peut rendre le dropout
   moins necessaire ;
2. elle **couple les exemples d'un meme batch** entre eux, ce qui la rend
   problematique pour les tres petits batchs (d'ou l'existence de la LayerNorm,
   utilisee dans les Transformers, qui normalise sur la dimension des features
   et non sur celle du batch).

## Online learning vs. mini-batch vs. full-batch training

In practice, it’s not always possible to train a model on the entire dataset at once — either due to memory limitations or the nature of the application (e.g., streaming data). This is where different training strategies come into play:


- Online Training (Stochastic Gradient Descent)

        - The model is updated one example at a time.

        - At each step, a single image is selected (often randomly), and the gradient is computed and used to update the model.

        - This introduces a lot of noise but can be useful when data arrives sequentially or memory is tight.

        - In theory converges faster, very fast per-step, but in practice not efficient in terms of computation time.
        


- Mini-Batch Training

        A middle ground: the training data is split into small batches of size B; the split changes at each epoch.

        The model is updated after computing the gradient on each batch.

        Offers a good trade-off between computational efficiency and gradient stability.

        Usually used in deep learning frameworks: it allows us a tradeoff in terms of memory requirement.
        
     

- Full-Batch Training

        The gradient is computed over the entire training set before each update.

        This produces the most accurate gradient estimate, but:

            Requires loading all the data at once.

            Can be very slow per update, especially for large datasets.


To fully explore different training strategies, compare them directly on your datatset.

**Your task:**

- Make a comparison in terms of computational efficiency to do one training epoch with: online, mini-batch (B=200) and full training.

- Then make a convergence speed comparison between these 3 training methods: how many epochs are needed to reach good performance ?

*NB: it is important to always adapt the learning rate to have nice performance!*

In [ ]:
## -----------> TODO <----------------
# 1) COUT D'UNE EPOCH. On mesure le temps d'une seule epoch pour chaque strategie.
#    Attention : online = 20 000 mises a jour, chacune sur 1 image ; full-batch =
#    1 mise a jour sur 20 000 images. Le VOLUME de calcul est le meme, mais pas
#    du tout le temps reel : le GPU/CPU est bien plus efficace sur de gros
#    tenseurs que sur 20 000 petits (cout fixe par appel Python + moins bonne
#    utilisation du materiel).
print("--- Temps pour UNE epoch ---")
temps_epoch = {}
for nom, bs in [("online (B=1)", 1), ("mini-batch (B=200)", 200), ("full-batch", None)]:
    th.manual_seed(0)
    m = make_mlp((100,), activation=nn.ReLU)
    h = run(m, lr=0.01 if bs == 1 else 0.1, momentum=0.9, NEpochs=1,
            batch_size=bs, plot=False)
    temps_epoch[nom] = h['time']
    nb_maj = {1: len(Xtrain), 200: len(Xtrain) // 200, None: 1}[bs]
    print(f"{nom:20s} | {nb_maj:6d} mises a jour | {h['time']:7.2f} s"
          f" | valid acc apres 1 epoch = {h['valid_acc'][-1]:.4f}")

L'online est de loin le plus lent en temps de calcul, alors que c'est celui
qui fait le plus de progres par epoch. C'est tout le compromis.

2) **VITESSE DE CONVERGENCE**. On compare maintenant a nombre d'epochs egal, en
adaptant le learning rate a chaque strategie (obligatoire : en online le gradient
est tres bruite, il faut un pas plus petit ; en full-batch le gradient est exact
mais on ne fait qu'un pas par epoch, il faut un pas plus grand).

In [ ]:
configs = [
    ("online (B=1, lr=0.01)",      1,    0.01),
    ("mini-batch (B=200, lr=0.1)", 200,  0.1),
    ("full-batch (lr=0.5)",        None, 0.5),
]
plt.figure(figsize=(11, 4))
for i, (nom, bs, lr) in enumerate(configs):
    th.manual_seed(0)
    m = make_mlp((100,), activation=nn.ReLU)
    NE = 10 if bs == 1 else 40          # l'online est trop lent pour 40 epochs
    h = run(m, lr=lr, momentum=0.9, NEpochs=NE, batch_size=bs, plot=False)
    plt.subplot(1, 2, 1); plt.plot(h['valid_acc'], label=nom)
    plt.subplot(1, 2, 2)
    t_par_epoch = h['time'] / NE
    plt.plot([t_par_epoch * k for k in range(NE)], h['valid_acc'], label=nom)
    print(f"{nom:28s} -> valid acc = {h['valid_acc'][-1]:.4f} en {h['time']:.1f} s ({NE} epochs)")
plt.subplot(1, 2, 1); plt.xlabel("epoch"); plt.ylabel("accuracy validation")
plt.title("A nombre d'epochs egal"); plt.legend(fontsize=7); plt.grid(alpha=.3)
plt.subplot(1, 2, 2); plt.xlabel("temps de calcul (s)")
plt.title("A temps de calcul egal"); plt.legend(fontsize=7); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Conclusion - et c'est la figure la plus importante du TP.**

- **Par epoch**, l'online gagne : il fait 20 000 mises a jour la ou le full-batch
  n'en fait qu'une.
- **Par seconde**, il perd largement : chaque mise a jour coute presque aussi cher
  qu'un batch entier a cause du surcout fixe.
- Le **mini-batch** est le meilleur des deux mondes, et c'est pour cela que
  c'est ce qu'on utilise systematiquement en pratique. Il beneficie de la
  vectorisation du materiel tout en faisant beaucoup de mises a jour, et le
  bruit residuel du gradient aide meme a sortir des mauvais minima.

Ordre de grandeur a retenir : le bruit du gradient d'un mini-batch decroit en
$1/\sqrt{B}$. Passer de B=1 a B=100 divise le bruit par 10 pour un cout par
mise a jour qui, lui, augmente beaucoup moins que 100.

## Dropout layer to prevent overfitting -- in more details

Even though Fashion-MNIST is a relatively simple dataset, it is still very easy to overfit, especially with small models and limited regularization.

To remind you (you discussed this in TD) overfitting means:

    Your model performs well on the training set (from TD - the fitted curve went through all the datapoints),

    But performs poorly on unseen data (from TD - the curve would change if you added new data, i.e., the fit (model) cannot be generalized).

Dropout is a regularization technique that randomly **drops** a fraction of neurons during training (e.g., 20% with p=0.2). This forces the model to not rely too much on any one feature or path!

- During training dropout randomly sets activations to zero with probability p.

- During evaluation dropout turns off, and the full network is used.

In the section on Deeper networks above, you already played with the dropout option. When applied after a single hidden layer, it helps prevent overfitting in a shallow model. If, however, it is applied after each hidden layer in a deeper network, it is addressing overfitting across more complex representations, where the risk of memorization is higher.

**Your task:**

- Code this modification (with p=0.2), rerun the training process and compare the training curves.

As stated, when you use a Dropout layer, the layer acts differently in the train mode and evaluation mode. You should take this into account when you train the model and when you compute the performance on the validation set.

In [ ]:
## -----------> TODO <----------------
# Le point cle demande ici : Dropout n'a PAS le meme comportement en train et
# en eval. Notre fonction evaluate() appelle bien model.eval(), et la boucle
# d'entrainement model.train(). Verifions concretement ce que ca change.
th.manual_seed(0)
model_dp = make_mlp((200, 200), activation=nn.ReLU, dropout=0.2)
hist_dp = run(model_dp, lr=0.1, momentum=0.9, NEpochs=60, batch_size=200,
              label="2 x 200 + Dropout(0.2)")

# Demonstration : deux passes en mode train donnent des resultats DIFFERENTS
# (des neurones differents sont desactives a chaque fois), alors qu'en mode
# eval le resultat est deterministe.
x = Xvalid[:1]
model_dp.train()
print("mode train, passe 1 :", model_dp(x)[0, :3].detach().numpy())
print("mode train, passe 2 :", model_dp(x)[0, :3].detach().numpy(), " <- different !")
model_dp.eval()
print("mode eval,  passe 1 :", model_dp(x)[0, :3].detach().numpy())
print("mode eval,  passe 2 :", model_dp(x)[0, :3].detach().numpy(), " <- identique")

# Et l'erreur classique : evaluer sans passer en eval() sous-estime le modele.
model_dp.train()
with th.no_grad():
    acc_train_mode = (model_dp(Xvalid).argmax(1) == Yvalid).float().mean().item()
model_dp.eval()
with th.no_grad():
    acc_eval_mode = (model_dp(Xvalid).argmax(1) == Yvalid).float().mean().item()
print(f"\naccuracy validation en mode train() : {acc_train_mode:.4f}  <- FAUX")
print(f"accuracy validation en mode eval()  : {acc_eval_mode:.4f}  <- correct")

**Pourquoi cette difference ?** Pendant l'entrainement, chaque neurone est
desactive avec probabilite $p$ ; l'esperance de la sortie est donc reduite d'un
facteur $(1-p)$. PyTorch compense immediatement en divisant les activations
survivantes par $(1-p)$ (c'est l'*inverted dropout*), de sorte qu'en inference
on n'ait **rien** a faire : `eval()` desactive simplement le dropout et utilise
tout le reseau.

L'intuition profonde : le dropout entraine implicitement un **ensemble** de
sous-reseaux qui partagent leurs poids, et l'inference avec le reseau complet
approxime la moyenne de cet ensemble. C'est de l'ensemble learning gratuit -
d'ou l'effet regularisant.

## Final Test

- Take two of your best models (among those with one and two hidden layers) and run the evaluation on the test set.
- Train the best two models with all the training data and compute the results on the test set.

In [ ]:
## -----------> TODO <----------------
# Le jeu de TEST n'a encore JAMAIS servi. C'est essentiel : tous les choix
# faits jusqu'ici (nb de couches, taille, dropout, lr, momentum) ont ete faits
# sur la VALIDATION. Le test ne sert qu'a estimer une derniere fois, sans biais,
# la performance du modele retenu. Si on l'utilisait pour choisir, on
# recommencerait a surapprendre - sur le test cette fois.
Xtest_n = Xtest / 255.0          # meme normalisation que le train
print("test :", Xtest_n.shape, Ytest.shape)

meilleurs = {
    "1 couche cachee (200, dropout 0.2)":       dict(sizes=(200,),     dropout=0.2),
    "2 couches cachees (200x200, dropout 0.2)": dict(sizes=(200, 200), dropout=0.2),
}

# --- Etape 1 : entrainement sur les 20 000 images du train seulement ---
print("\n=== Entraines sur 20 000 images ===")
for nom, cfg in meilleurs.items():
    th.manual_seed(0)
    m = make_mlp(cfg["sizes"], activation=nn.ReLU, dropout=cfg["dropout"], batchnorm=True)
    h = run(m, lr=0.1, momentum=0.9, NEpochs=60, batch_size=200, plot=False)
    l_te, a_te = evaluate(m, Xtest_n, Ytest, loss_function)
    print(f"{nom:42s} | valid {h['valid_acc'][-1]:.4f} | TEST {a_te:.4f}")

# --- Etape 2 : re-entrainement sur TOUTES les donnees (train + validation) ---
# Une fois les hyperparametres figes, on a interet a reentrainer sur le maximum
# de donnees : 30 000 images au lieu de 20 000. On n'a plus de validation pour
# faire de l'early stopping, donc on garde le nombre d'epochs trouve avant.
Xfull, Yfull = allXtrain[:30000], allYtrain[:30000]
print(f"\n=== Entraines sur {len(Xfull)} images (train + validation) ===")
modeles_finaux, resultats_finaux = {}, {}
for nom, cfg in meilleurs.items():
    th.manual_seed(0)
    m = make_mlp(cfg["sizes"], activation=nn.ReLU, dropout=cfg["dropout"], batchnorm=True)
    opt = th.optim.SGD(m.parameters(), lr=0.1, momentum=0.9)
    train_model(m, loss_function, opt, Xfull, Yfull, 60,
                Xvalid=Xtest_n, Yvalid=Ytest,   # ici "valid" ne sert qu'au suivi
                batch_size=200, plot=False, verbose=False)
    l_te, a_te = evaluate(m, Xtest_n, Ytest, loss_function)
    modeles_finaux[nom], resultats_finaux[nom] = m, a_te
    print(f"{nom:42s} | TEST {a_te:.4f}")

meilleur_nom = max(resultats_finaux, key=resultats_finaux.get)
meilleur = modeles_finaux[meilleur_nom]
print(f"\nMeilleur modele : {meilleur_nom} -> {resultats_finaux[meilleur_nom]:.4f} sur le test")

### Matrice de confusion

L'accuracy globale cache *ou* le modele se trompe. La matrice de confusion le dit :
la case (i, j) compte les images de classe i predites comme j. Une bonne matrice
est diagonale.

In [ ]:
meilleur.eval()
with th.no_grad():
    pred_test = meilleur(Xtest_n).argmax(dim=1)

C = th.zeros(10, 10, dtype=th.long)
for vrai, predit in zip(Ytest, pred_test):
    C[vrai, predit] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(classlist, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(classlist, fontsize=8)
ax.set_xlabel("classe predite"); ax.set_ylabel("classe reelle")
ax.set_title("Matrice de confusion sur le jeu de test")
for i in range(10):
    for j in range(10):
        if C[i, j] > 0:
            ax.text(j, i, int(C[i, j]), ha="center", va="center", fontsize=6,
                    color="white" if C[i, j] > C.max() / 2 else "black")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

print("\nAccuracy par classe :")
for c in range(10):
    masque = (Ytest == c)
    print(f"  {classlist[c]:15s} : {(pred_test[masque] == c).float().mean().item():.3f}")

### Conclusion du TP

Chemin parcouru, en accuracy de validation :

| Modele | Accuracy |
|---|---|
| Hasard | 0.10 |
| Lineaire (regression logistique) | ~0.83 |
| 1 couche cachee 50, ReLU + momentum | ~0.86 |
| 2 couches cachees 200, ReLU + BatchNorm + Dropout, mini-batch | ~0.89 |

Les confusions restantes sont presque toutes entre **chemise / pull / manteau /
t-shirt** : ces quatre classes ont des silhouettes tres proches et se
distinguent par la **texture** et les **details locaux**. Or un MLP applique a
une image aplatie de 784 pixels perd toute la structure spatiale : pour lui, deux
pixels voisins n'ont aucune raison d'etre relies, et il doit reapprendre un motif
separement pour chaque position possible.

C'est exactement ce que les **convolutions** resolvent (voisinage local +
partage des poids), et donc l'objet du TP suivant.